# E-commerce Sales Analytics
## Data Cleaning and Exploratory Data Analysis (EDA)

This notebook inspects data quality and analyzes sales by month, country, product, and customer.

**Dataset required:** `Ecommerce_Sales_Data.xlsx`

Upload the Excel file to the Colab working directory before running the analysis.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load dataset
file_path = Path('Ecommerce_Sales_Data.xlsx')
if not file_path.exists():
    raise FileNotFoundError(
        'Ecommerce_Sales_Data.xlsx was not found. Upload the Excel file to the Colab working directory.'
    )

df = pd.read_excel(file_path)

required_columns = [
    'InvoiceNo', 'StockCode', 'Description', 'Quantity',
    'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Total Sales'
]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

print('DATASET OVERVIEW')
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print('\nData types and non-null counts:')
df.info()
display(df.head())

## 1. Data Quality Checks

Review missing values, duplicate rows, and negative values before deciding whether any records should be removed or changed.

In [ ]:
missing_summary = df.isna().sum().to_frame(name='Missing Count')
missing_summary['Missing Percentage (%)'] = (missing_summary['Missing Count'] / len(df) * 100).round(2)
display(missing_summary)

duplicate_count = int(df.duplicated().sum())
print(f'Duplicate rows detected: {duplicate_count:,}')

print('\nNegative values:')
for col in ['Quantity', 'UnitPrice', 'Total Sales']:
    print(f'{col}: {(pd.to_numeric(df[col], errors="coerce") < 0).sum():,}')

# Missing values are not automatically replaced with zero,
# and duplicate rows are not automatically deleted.

## 2. Data Cleaning and Feature Engineering

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

for col in ['Quantity', 'UnitPrice', 'Total Sales', 'CustomerID']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['InvoiceNo'] = df['InvoiceNo'].astype('string').str.strip()
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype('string')
df['IsCancellation'] = df['InvoiceNo'].str.startswith('C', na=False)

print('Data types after conversion:')
print(df.dtypes)
print(f"\nCancellation invoice rows: {df['IsCancellation'].sum():,}")
print(f"Rows with missing CustomerID: {df['CustomerID'].isna().sum():,}")

## 3. Key Business Metrics

Revenue is calculated from `Total Sales`. AOV is defined here as total net sales (including negative adjustments present in the data) divided by the number of unique invoices not beginning with `C`. Interpret this definition when presenting results.

In [ ]:
valid_sales = df.dropna(subset=['InvoiceNo', 'Total Sales']).copy()
total_revenue = valid_sales['Total Sales'].sum()
sales_invoices = valid_sales.loc[~valid_sales['IsCancellation'], 'InvoiceNo'].nunique()
aov = total_revenue / sales_invoices if sales_invoices else np.nan

print(f'Total Revenue (net): {total_revenue:,.2f}')
print(f'Unique Sales Invoices: {sales_invoices:,}')
print(f'Average Order Value (AOV): {aov:,.2f}')
print(f"Unique Customers: {df['CustomerID'].nunique():,}")
print(f"Unique Products: {df['StockCode'].nunique():,}")
print(f"Countries: {df['Country'].nunique():,}")

## 4. Monthly Sales Analysis

In [ ]:
monthly_sales = (
    valid_sales.dropna(subset=['YearMonth'])
    .groupby('YearMonth', as_index=False)
    .agg(Total_Sales=('Total Sales', 'sum'), Invoice_Count=('InvoiceNo', 'nunique'))
    .sort_values('YearMonth')
)
display(monthly_sales)

if not monthly_sales.empty:
    best_month = monthly_sales.loc[monthly_sales['Total_Sales'].idxmax()]
    print(f"Highest sales month: {best_month['YearMonth']}")
    print(f"Sales in that month: {best_month['Total_Sales']:,.2f}")

## 5. Sales by Country

In [ ]:
country_sales = (
    valid_sales.groupby('Country', as_index=False)
    .agg(
        Total_Sales=('Total Sales', 'sum'),
        Invoice_Count=('InvoiceNo', 'nunique'),
        Unique_Customers=('CustomerID', 'nunique')
    )
    .sort_values('Total_Sales', ascending=False)
)
display(country_sales.head(10))

## 6. Top Products by Sales

In [ ]:
product_sales = (
    valid_sales.groupby(['StockCode', 'Description'], dropna=False, as_index=False)
    .agg(Total_Sales=('Total Sales', 'sum'), Total_Quantity=('Quantity', 'sum'))
    .sort_values('Total_Sales', ascending=False)
)
display(product_sales.head(10))

## 7. Customer Analysis

In [ ]:
customer_sales = (
    valid_sales.dropna(subset=['CustomerID'])
    .groupby('CustomerID', as_index=False)
    .agg(Total_Sales=('Total Sales', 'sum'), Invoice_Count=('InvoiceNo', 'nunique'))
    .sort_values('Total_Sales', ascending=False)
)
display(customer_sales.head(10))

## 8. Final Data Quality Summary

Review these checks before deciding whether to remove duplicates or impute missing data.

In [ ]:
print(f'Rows in dataset: {len(df):,}')
print(f'Duplicate rows detected: {duplicate_count:,}')
print(f"Rows with missing InvoiceDate: {df['InvoiceDate'].isna().sum():,}")
print(f"Rows with missing Total Sales: {df['Total Sales'].isna().sum():,}")
print(f"Rows with missing CustomerID: {df['CustomerID'].isna().sum():,}")
print('\nData cleaning and EDA completed.')
print('Review data quality findings before removing or imputing data.')